# Strategy - Correlation Aware BTC RS

# ============================================================
# S12 - BTC RS + Correlation Risk Scaling
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    close = xr.where(
        is_liquid == 1,
        close,
        np.nan
    )

    btc = data.sel(
        field="close",
        asset="BTC"
    )

    # --------------------------------------------------------
    # BTC Regime Filter
    # --------------------------------------------------------

    btc_sma200 = qnta.sma(
        btc,
        200
    )

    bull_market = xr.where(
        btc > btc_sma200,
        1,
        0
    )

    # --------------------------------------------------------
    # Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # BTC Relative Strength
    # --------------------------------------------------------

    momentum = (
        close /
        close.shift(time=21)
        - 1
    )

    btc_momentum = (
        btc /
        btc.shift(time=21)
        - 1
    )

    btc_rs = (
        momentum
        - btc_momentum
    )

    score = btc_rs.rank(
        "asset"
    )

    # --------------------------------------------------------
    # Select Leaders
    # --------------------------------------------------------

    ranks = score.rank(
        "asset"
    )

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility
    # --------------------------------------------------------

    inv_vol = (
        1 /
        (vol20 + 1e-6)
    )

    weights = weights * inv_vol

    weights = weights / (
        weights.sum("asset")
        + 1e-6
    )

    # --------------------------------------------------------
    # Correlation Risk Scaling
    # --------------------------------------------------------

    dispersion = ret.std("asset")

    dispersion_ma60 = qnta.sma(
        dispersion,
        60
    )

    dispersion_ratio = (
        dispersion /
        (dispersion_ma60 + 1e-6)
    )

    risk_multiplier = xr.where(
        dispersion_ratio > 1.05,
        1.0,
        xr.where(
            dispersion_ratio > 0.95,
            0.75,
            0.50
        )
    )

    # --------------------------------------------------------
    # Apply Filters
    # --------------------------------------------------------

    weights = (
        weights
        * risk_multiplier
        * bull_market
    )

    return weights.fillna(0)

# ============================================================
# Generate Weights
# ============================================================

weights = strategy(data)

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

qnout.write(weights)

100% (15262844 of 15262844) |############| Elapsed Time: 0:00:00 Time:  0:00:000:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field            equity  relative_return  volatility  underwater  \
time                                                               
2026-06-06  4016.286222              0.0    0.547079   -0.178383   
2026-06-07  4016.286222              0.0    0.547007   -0.178383   
2026-06-08  4016.286222              0.0    0.546936   -0.178383   
2026-06-09  4016.286222              0.0    0.546865   -0.178383   
2026-06-10  4016.286222              0.0    0.546794   -0.178383   

field       max_drawdown  sharpe_ratio  mean_return  bias  instruments  \
time                                                                     
2026-06-06     -0.692999      1.716072     0.938826   0.0         31.0   
2026-06-07     -0.692999      1.715845     0.938580   0.0         31.0   
2026-06-08     -0.692999      1.715618     0.938333   0.0         31.0   
2026-06-09     -0.692999      1.715391     0.938087   0.0         31.0   
2026-06-10     -0.692999      1.715165     0.937841   0.0         31.0   

field       avg_turnover  avg_holding_time  
time                                        
2026-06-06      0.234529          3.736629  
2026-06-07      0.234468          3.736629  
2026-06-08      0.234406          3.736629  
2026-06-09      0.234345          3.736629  
2026-06-10      0.234283          3.736629  

Final Metrics:
field
equity              4016.286222
sharpe_ratio           1.715165
max_drawdown          -0.692999
avg_turnover           0.234283
avg_holding_time       3.736629
Name: 2026-06-10 00:00:00, dtype: float64
